# Normative Model (Structural)

Train a quantile regression MLP that predicts the 10th, 50th and 90th percentiles of the total gray matter volume (`TotalGrayVol`) given Age and Sex.

Input data: `../data/processed/abide_structural_cleaned.csv` (from `preprocessing_structural.ipynb`)

In [1]:
# Import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
import os
import json
import random
import copy

In [2]:
# Random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device: GPU (cuda), Apple Silicon (mps) or cpu
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

Device: mps
PyTorch version: 2.5.1


## Load Data

Loading the preprocessed dataset and the scaling parameters (used later to transform predictions back to mm³).

In [3]:
# Preprocessed dataset
df = pd.read_csv("../data/processed/abide_structural_cleaned.csv")
print(f"Total samples: {len(df)}")

# Scaling parameters
with open("../data/processed/scaling_params_structural.json", "r") as f:
    scaling_params = json.load(f)

age_mean = scaling_params['age_mean']
age_std = scaling_params['age_std']
vol_mean = scaling_params['vol_mean']
vol_std = scaling_params['vol_std']
print(scaling_params)

df.head()

Total samples: 477
{'age_mean': 16.95909895178197, 'age_std': 7.32014240993593, 'vol_mean': 726986.6218631573, 'vol_std': 77565.06628857981}


,FILE_ID,SITE_ID,AGE_AT_SCAN,SEX,TotalGrayVol,IntraCranialVol,Left-Hippocampus,Right-Hippocampus,Hippocampus,Age_scaled,TotalGrayVol_scaled
0,Pitt_0050030,PITT,25.12,0,706581.114475,1.157527e+06,3982.0,4061.0,8043.0,1.114856,-0.263076
1,Pitt_0050031,PITT,12.92,0,783889.187268,1.208583e+06,4235.0,4477.0,8712.0,-0.551779,0.733611
2,Pitt_0050032,PITT,19.80,0,730058.146235,1.206391e+06,4475.0,4598.0,9073.0,0.388094,0.039599
3,Pitt_0050033,PITT,12.15,0,720818.859215,1.059904e+06,4445.0,4379.0,8824.0,-0.656968,-0.079517
4,Pitt_0050034,PITT,14.77,0,729693.739442,1.094946e+06,4074.0,4394.0,8468.0,-0.299051,0.034901


**Dataset and DataLoader**

Converting the inputs (`Age_scaled`, `SEX`) and target (`TotalGrayVol_scaled`) into a PyTorch `TensorDataset`, and splitting the data into training (70%), validation (15%) and testing (15%) sets.

In [4]:
# Inputs and target as tensors
X = torch.tensor(df[['Age_scaled', 'SEX']].values, dtype=torch.float32)
y = torch.tensor(df['TotalGrayVol_scaled'].values, dtype=torch.float32).unsqueeze(1)

# call dataset
dataset = TensorDataset(X, y)

# Split data (70/15/15)
generator = torch.Generator().manual_seed(SEED)
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=generator)

print(f"Training set size: {len(train_dataset)} samples")
print(f"Validation set size: {len(val_dataset)} samples")
print(f"Test set size: {len(test_dataset)} samples")

# dataloaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Training set size: 333 samples
Validation set size: 71 samples
Test set size: 73 samples


## Model

MLP with a shared backbone and three outputs: the 10th, 50th (median) and 90th percentiles.

To avoid quantile crossing (e.g. $q_{10} > q_{50}$), the model predicts the median and two distances, made positive with `F.softplus`:

$$\hat{y}_{10} = \hat{y}_{50} - \text{Softplus}(\Delta_{10}) \qquad \hat{y}_{90} = \hat{y}_{50} + \text{Softplus}(\Delta_{90})$$

So $\hat{y}_{10} \le \hat{y}_{50} \le \hat{y}_{90}$ always.

Where:
* $\Delta_{10}$ and $\Delta_{90}$ are the raw outputs of `head_lower` and `head_upper`. They can be any number (positive or negative).
* $\text{Softplus}(x) = \log(1 + e^x)$ turns any number into a positive one (a smooth version of ReLU).

So $\text{Softplus}(\Delta_{10})$ is the distance from the median down to $q_{10}$, and $\text{Softplus}(\Delta_{90})$ the distance up to $q_{90}$.

In [5]:
class QuantileMLP(nn.Module):
    def __init__(self, in_features=2, hidden_dim=32):
        super().__init__()
        # Shared backbone
        self.backbone = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        # Outputs: median and the two distances
        self.head_q50 = nn.Linear(hidden_dim, 1)
        self.head_lower = nn.Linear(hidden_dim, 1)
        self.head_upper = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        features = self.backbone(x)
        q50 = self.head_q50(features)
        # softplus makes the distances positive
        q10 = q50 - F.softplus(self.head_lower(features))
        q90 = q50 + F.softplus(self.head_upper(features))
        # The output has a shape [batch_size, 3], with columns q10, q50, q90
        return torch.cat([q10, q50, q90], dim=1)

model = QuantileMLP().to(device)
print(model)
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")

# check with the first batch from train_loader (untrained model)
batch_X, batch_y = next(iter(train_loader))
with torch.no_grad():
    preds = model(batch_X.to(device))
print(f"Output shape: {preds.shape}")
print(f"First 3 predictions [q10, q50, q90]:\n{preds[:3]}")

QuantileMLP(
  (backbone): Sequential(
    (0): Linear(in_features=2, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
  )
  (head_q50): Linear(in_features=32, out_features=1, bias=True)
  (head_lower): Linear(in_features=32, out_features=1, bias=True)
  (head_upper): Linear(in_features=32, out_features=1, bias=True)
)
Number of parameters: 1251


Output shape: torch.Size([16, 3])


First 3 predictions [q10, q50, q90]:
tensor([[-0.7158,  0.0237,  0.6584],
        [-0.7088,  0.0241,  0.6817],
        [-0.7398, -0.0150,  0.6511]], device='mps:0')
